# Laboratorio: ¿cuánto cambian recall y precisión al repartir por tiempo en lugar de al azar?

**Pregunta:** ¿cuánto cambian el recall y la precisión de una regresión logística en detección de intrusiones con reparto por tiempo en lugar de al azar?

**Hipótesis:** ambas bajan con reparto por tiempo.

**Diseño (comparación pareada):**
- Mismo dataset, mismas variables, mismo modelo (regresión logística), mismo umbral (0,5).
- Lo único que cambia es **cómo se reparte** entre entrenamiento y prueba.
- Para medir variabilidad: 5 "pliegues" temporales de origen móvil (se entrena con todo el pasado y se prueba con el bloque siguiente). Para cada pliegue, el reparto al azar usa **los mismos datos y los mismos tamaños**, repetido varias veces con semillas distintas.

**Después de medir, el laboratorio busca el porqué:**
1. ¿Hay "gemelas" (filas idénticas o casi) entre entrenamiento y prueba? ¿Más en el reparto al azar?
2. ¿Qué tipos de ataque se pierden con el reparto por tiempo?
3. ¿Aprende atajos (IP)?

**Ojo con una trampa de interpretación:** el reparto por tiempo cambia dos cosas a la vez: (a) quita las gemelas (fuga de datos) y (b) introduce cambios reales entre pasado y futuro (ataques nuevos, otra proporción de ataques). Las secciones de diagnóstico intentan separar ambos efectos.

## 0. Configuración (decide esto ANTES de ejecutar)

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             accuracy_score, average_precision_score)
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---- Parámetros del experimento ------------------------------------------
DATA_DIR = Path("data/UNSW-NB15")  # carpeta con UNSW-NB15_1.csv ... UNSW-NB15_4.csv
SAMPLE_FRAC = 0.25      # 0.25 para probar rápido; 1.0 para el resultado final
N_BLOCKS = 6            # 6 bloques temporales -> 5 pliegues
N_RANDOM_REPEATS = 10   # repeticiones del reparto al azar por pliegue
THRESHOLD = 0.5         # umbral fijo (estudiarlo es tarea del capítulo siguiente)
BIG_DIFF = 0.05         # "diferencia grande" = 5 puntos porcentuales. Fíjala antes de mirar resultados.
MIN_FOLDS_AGREE = 4     # nº de pliegues (de 5) en los que debe cumplirse la caída para dar la hipótesis por confirmada
SEED = 42

## 1. Carga y limpieza

Dataset recomendado: **UNSW-NB15** (los 4 CSV "en bruto": `UNSW-NB15_1.csv` ... `UNSW-NB15_4.csv`).
Motivos: tiene `Stime`/`Ltime` como marcas de tiempo Unix, trae IP de origen y destino (necesarias para la prueba del atajo) y no tiene los problemas conocidos de marcas de tiempo del CIC-IDS2017 (horas en formato 12 h sin AM/PM).
Los CSV no traen cabecera; los nombres de columna van en el código.

In [ ]:
COLS = ["srcip","sport","dstip","dsport","proto","state","dur","sbytes","dbytes","sttl","dttl",
        "sloss","dloss","service","Sload","Dload","Spkts","Dpkts","swin","dwin","stcpb","dtcpb",
        "smeansz","dmeansz","trans_depth","res_bdy_len","Sjit","Djit","Stime","Ltime","Sintpkt",
        "Dintpkt","tcprtt","synack","ackdat","is_sm_ips_ports","ct_state_ttl","ct_flw_http_mthd",
        "is_ftp_login","ct_ftp_cmd","ct_srv_src","ct_srv_dst","ct_dst_ltm","ct_src_ltm",
        "ct_src_dport_ltm","ct_dst_sport_ltm","ct_dst_src_ltm","attack_cat","Label"]
assert len(COLS) == 49

ID_COLS    = ["srcip", "sport", "dstip", "dsport"]   # identificadores: fuera del modelo base
TIME_COLS  = ["Stime", "Ltime"]                      # el tiempo sirve para repartir, NO como variable
LABEL_COLS = ["Label", "attack_cat"]
CAT_COLS   = ["proto", "state", "service"]
NUM_COLS   = [c for c in COLS if c not in ID_COLS + TIME_COLS + LABEL_COLS + CAT_COLS]
FEAT_COLS  = NUM_COLS + CAT_COLS


def load_unsw(data_dir):
    files = sorted(Path(data_dir).glob("UNSW-NB15_[1-4].csv"))
    assert len(files) == 4, f"Esperaba 4 ficheros UNSW-NB15_[1-4].csv en {data_dir}, encontré {len(files)}"
    return pd.concat([pd.read_csv(f, header=None, names=COLS, low_memory=False) for f in files],
                     ignore_index=True)


df = load_unsw(DATA_DIR)
print("Filas cargadas:", len(df))

# --- limpieza ---
for c in ["sport", "dsport", "ct_ftp_cmd", "ct_flw_http_mthd", "is_ftp_login", "Stime", "Ltime"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CAT_COLS + ["srcip", "dstip"]:
    df[c] = df[c].astype(str).str.strip()
df["attack_cat"] = df["attack_cat"].fillna("Normal").astype(str).str.strip().replace({"": "Normal", "Backdoors": "Backdoor"})
df["Label"] = pd.to_numeric(df["Label"], errors="coerce").astype(int)
df = df.dropna(subset=["Stime"])

# --- orden temporal (la base de todo el experimento) ---
df = df.sort_values("Stime", kind="mergesort").reset_index(drop=True)

if SAMPLE_FRAC < 1.0:   # submuestreo uniforme: conserva el orden temporal
    df = df.sample(frac=SAMPLE_FRAC, random_state=SEED).sort_values("Stime", kind="mergesort").reset_index(drop=True)

y = df["Label"].astype(int)
X = df[FEAT_COLS]
n = len(df)
print(f"Filas usadas: {n:,} | ataques: {y.sum():,} ({y.mean():.1%})")

### 1.1 Comprobación de las marcas de tiempo
Antes de fiarnos del reparto por tiempo hay que ver que el tiempo tiene sentido: rango de fechas, huecos y proporción de ataques por bloque.

In [ ]:
t = pd.to_datetime(df["Stime"], unit="s")
print("Desde:", t.min(), "| Hasta:", t.max())
gap = t.diff().dt.total_seconds().fillna(0)
print("Mayor hueco entre filas consecutivas: %.1f horas (en %s)" % (gap.max() / 3600, t[gap.idxmax()]))
print("Stime ordenado:", df["Stime"].is_monotonic_increasing)

edges = np.linspace(0, n, N_BLOCKS + 1).astype(int)
blocks = pd.DataFrame({
    "bloque": range(N_BLOCKS),
    "desde": [t.iloc[edges[i]] for i in range(N_BLOCKS)],
    "hasta": [t.iloc[edges[i + 1] - 1] for i in range(N_BLOCKS)],
    "filas": np.diff(edges),
    "% ataques": [y.iloc[edges[i]:edges[i + 1]].mean() * 100 for i in range(N_BLOCKS)],
}).round(2)
blocks

## 2. Piezas del experimento
Modelo, métricas y los dos tipos de reparto. Con el mismo tamaño de entrenamiento y de prueba en ambos repartos.

In [ ]:
def log1p_pos(a):
    return np.log1p(np.clip(a, 0, None))


def make_model(num_cols, cat_cols, min_freq=100):
    # Regresión logística con preprocesado mínimo (log + estandarizar + one-hot).
    parts = []
    if num_cols:
        parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                       ("log", FunctionTransformer(log1p_pos)),
                                       ("sc", StandardScaler())]), list(num_cols)))
    if cat_cols:
        parts.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                           min_frequency=min_freq), list(cat_cols)))
    return Pipeline([("prep", ColumnTransformer(parts)),
                     ("clf", LogisticRegression(max_iter=1000))])


def metrics(y_true, proba, thr=THRESHOLD):
    pred = (proba >= thr).astype(int)
    return dict(
        accuracy=accuracy_score(y_true, pred),
        precision=precision_score(y_true, pred, zero_division=0),
        recall=recall_score(y_true, pred, zero_division=0),
        f1=f1_score(y_true, pred, zero_division=0),
        pr_auc=average_precision_score(y_true, proba),
        prevalence=float(np.mean(y_true)),
        acc_todo_normal=1 - float(np.mean(y_true)),   # accuracy del modelo tonto "todo normal"
    )


def temporal_split(n_total, train_end, test_end):
    return np.arange(0, train_end), np.arange(train_end, test_end)


def random_split(n_total, train_end, test_end, seed):
    # Mismo conjunto de datos (primeras test_end filas) y mismos tamaños que el reparto temporal,
    # pero mezclando al azar.
    rng = np.random.default_rng(seed)
    pool = rng.permutation(test_end)
    return pool[:train_end], pool[train_end:test_end]


def fit_predict(cols_num, cols_cat, tr, te, data=None):
    data = df if data is None else data
    m = make_model(cols_num, cols_cat)
    cols = list(cols_num) + list(cols_cat)
    m.fit(data.iloc[tr][cols], y.iloc[tr])
    return m.predict_proba(data.iloc[te][cols])[:, 1]

## 3. Experimento principal: azar vs tiempo en 5 pliegues
Para cada pliegue `k`: entrenar con los bloques `0..k` y probar con el bloque `k+1`.
El reparto al azar usa las mismas filas y los mismos tamaños, repetido `N_RANDOM_REPEATS` veces.

In [ ]:
rows = []
t0 = time.time()
for k in range(N_BLOCKS - 1):
    train_end, test_end = edges[k + 1], edges[k + 2]

    # --- por tiempo ---
    tr, te = temporal_split(n, train_end, test_end)
    p = fit_predict(NUM_COLS, CAT_COLS, tr, te)
    rows.append(dict(fold=k, split="tiempo", rep=0, n_train=len(tr), n_test=len(te), **metrics(y.iloc[te].values, p)))

    # --- al azar (repetido) ---
    for r in range(N_RANDOM_REPEATS):
        tr, te = random_split(n, train_end, test_end, seed=SEED + 1000 * k + r)
        p = fit_predict(NUM_COLS, CAT_COLS, tr, te)
        rows.append(dict(fold=k, split="azar", rep=r, n_train=len(tr), n_test=len(te), **metrics(y.iloc[te].values, p)))
    print(f"Pliegue {k} listo ({time.time() - t0:.0f} s)")

res = pd.DataFrame(rows)
METRICS = ["accuracy", "precision", "recall", "f1", "pr_auc"]
summary = (res.groupby(["fold", "split"])[METRICS + ["prevalence", "acc_todo_normal"]].mean()
              .round(4).unstack("split"))
summary

### 3.1 Diferencia pareada (tiempo − azar) y veredicto sobre la hipótesis
Un valor negativo significa que el reparto por tiempo da una nota **más baja**.

In [ ]:
rand_mean = res[res.split == "azar"].groupby("fold")[METRICS].mean()
rand_std  = res[res.split == "azar"].groupby("fold")[METRICS].std()
rand_min  = res[res.split == "azar"].groupby("fold")[METRICS].min()
rand_max  = res[res.split == "azar"].groupby("fold")[METRICS].max()
temp      = res[res.split == "tiempo"].set_index("fold")[METRICS]
delta = (temp - rand_mean)

print("Diferencia (tiempo - azar) por pliegue:")
display(delta.round(4))

print("\nVariabilidad del reparto al azar (desviación típica entre repeticiones):")
display(rand_std.round(4))

def veredicto(metric):
    d = delta[metric]
    n_drop = int((d < -BIG_DIFF).sum())
    n_up = int((d > BIG_DIFF).sum())
    fuera = int((temp[metric] < rand_min[metric]).sum())   # tiempo por debajo de TODAS las repeticiones al azar
    media = d.mean()
    if media < -BIG_DIFF and n_drop >= MIN_FOLDS_AGREE:
        v = "BAJA (hipótesis confirmada)"
    elif media > BIG_DIFF and n_up >= MIN_FOLDS_AGREE:
        v = "SUBE (hipótesis refutada)"
    else:
        v = "SIN CAMBIO CLARO / INCONSISTENTE"
    print(f"{metric:>9}: media Δ = {media:+.3f} | pliegues con caída > {BIG_DIFF:.2f}: {n_drop}/{len(d)} "
          f"| tiempo < mínimo del azar en {fuera}/{len(d)} -> {v}")

print("\nVEREDICTO")
for m in ["recall", "precision"]:
    veredicto(m)
print("\n(Secundarias)")
for m in ["f1", "pr_auc", "accuracy"]:
    veredicto(m)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, ["recall", "precision", "accuracy"]):
    for k in range(N_BLOCKS - 1):
        v = res[(res.fold == k) & (res.split == "azar")][m]
        ax.scatter([k] * len(v), v, color="grey", alpha=0.5, s=18, label="azar (repeticiones)" if k == 0 else None)
        ax.scatter(k, temp.loc[k, m], color="crimson", s=70, marker="D", label="tiempo" if k == 0 else None)
    ax.set_title(m); ax.set_xlabel("pliegue"); ax.set_xticks(range(N_BLOCKS - 1)); ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout(); plt.show()

### 3.2 La trampa del accuracy
Comparación del accuracy del modelo con el del modelo tonto que dice «todo normal». Si el accuracy apenas supera al tonto, el accuracy no está diciendo nada.

In [ ]:
acc = res.groupby(["fold", "split"])[["accuracy", "acc_todo_normal", "prevalence", "recall"]].mean().round(4)
acc

## 4. ¿Por qué cambia? Diagnósticos

Para los diagnósticos se usa un único reparto 70 % / 30 % (por tiempo y al azar, mismos tamaños).

In [ ]:
train_end, test_end = int(n * 0.7), n
tr_t, te_t = temporal_split(n, train_end, test_end)
tr_r, te_r = random_split(n, train_end, test_end, seed=SEED)

proba_t = fit_predict(NUM_COLS, CAT_COLS, tr_t, te_t)
proba_r = fit_predict(NUM_COLS, CAT_COLS, tr_r, te_r)
m_t, m_r = metrics(y.iloc[te_t].values, proba_t), metrics(y.iloc[te_r].values, proba_r)
pd.DataFrame({"azar": m_r, "tiempo": m_t}).round(4)

### 4.1 Gemelas exactas
¿Qué porcentaje de las filas de prueba tiene una fila **idéntica** (en todas las variables del modelo) en entrenamiento?

In [ ]:
h = pd.util.hash_pandas_object(df[FEAT_COLS], index=False).values

def twin_share(tr, te):
    train_hashes = set(h[tr])
    is_twin = np.array([x in train_hashes for x in h[te]])
    return is_twin

tw_t, tw_r = twin_share(tr_t, te_t), twin_share(tr_r, te_r)
yt, yr = y.iloc[te_t].values, y.iloc[te_r].values

tabla = pd.DataFrame({
    "azar":   {"% filas de prueba con gemela exacta": tw_r.mean() * 100,
               "% de los ATAQUES con gemela exacta": tw_r[yr == 1].mean() * 100},
    "tiempo": {"% filas de prueba con gemela exacta": tw_t.mean() * 100,
               "% de los ATAQUES con gemela exacta": tw_t[yt == 1].mean() * 100},
}).round(1)
tabla

### 4.2 Gemelas aproximadas: distancia al vecino más cercano
Distancia (en espacio de variables numéricas estandarizadas) de cada fila de prueba a su fila de entrenamiento más parecida. Si en el reparto al azar las distancias son mucho más pequeñas, el modelo se examina con conocidas.

In [ ]:
def nn_distances(tr, te, max_train=100_000, max_test=5_000, seed=SEED):
    rng = np.random.default_rng(seed)
    tr_s = rng.choice(tr, size=min(max_train, len(tr)), replace=False)
    te_s = rng.choice(te, size=min(max_test, len(te)), replace=False)
    pre = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("log", FunctionTransformer(log1p_pos)),
                    ("sc", StandardScaler())]).fit(df.iloc[tr][NUM_COLS])
    A, B = pre.transform(df.iloc[tr_s][NUM_COLS]), pre.transform(df.iloc[te_s][NUM_COLS])
    nn = NearestNeighbors(n_neighbors=1, algorithm="brute", n_jobs=-1).fit(A)
    d, _ = nn.kneighbors(B)
    return d.ravel(), y.iloc[te_s].values

d_r, ys_r = nn_distances(tr_r, te_r)
d_t, ys_t = nn_distances(tr_t, te_t)

print("Distancia mediana al vecino más cercano de entrenamiento")
print(f"  azar  : todas={np.median(d_r):.3f} | ataques={np.median(d_r[ys_r == 1]):.3f}")
print(f"  tiempo: todas={np.median(d_t):.3f} | ataques={np.median(d_t[ys_t == 1]):.3f}")

plt.figure(figsize=(7, 4))
bins = np.linspace(0, np.percentile(np.r_[d_r, d_t], 99), 60)
plt.hist(d_r, bins=bins, alpha=0.6, label="azar", density=True)
plt.hist(d_t, bins=bins, alpha=0.6, label="tiempo", density=True)
plt.xlabel("distancia al vecino más cercano en entrenamiento"); plt.legend(); plt.grid(alpha=0.3); plt.show()

### 4.3 ¿Cuánto de la "nota inflada" son gemelas exactas?
Se evalúa el modelo del reparto al azar **solo con las filas de prueba que NO tienen gemela exacta**. Si la nota baja hacia la del reparto por tiempo, buena parte de la inflación era fuga de datos. Si no baja, el cambio por tiempo se explica más por cambios reales entre pasado y futuro.

In [ ]:
mask = ~tw_r
res_nodup = metrics(yr[mask], proba_r[mask])
pd.DataFrame({
    "azar (todo)": m_r,
    "azar (sin gemelas exactas)": res_nodup,
    "tiempo": m_t,
}).loc[["precision", "recall", "f1", "pr_auc", "prevalence"]].round(4)

### 4.4 ¿Qué ataques se pierden por tiempo?
Recall por tipo de ataque en cada reparto, y cuántos ejemplos de cada tipo vio el modelo en entrenamiento. Un tipo que aparece en la prueba y casi no en entrenamiento es cambio real (deriva), no fuga de datos.

In [ ]:
cat_all = df["attack_cat"]

def recall_by_cat(te, proba):
    pred = (proba >= THRESHOLD).astype(int)
    d = pd.DataFrame({"cat": cat_all.iloc[te].values, "y": y.iloc[te].values, "pred": pred})
    d = d[d.y == 1]
    return d.groupby("cat")["pred"].mean()

out = pd.DataFrame({
    "recall azar": recall_by_cat(te_r, proba_r),
    "recall tiempo": recall_by_cat(te_t, proba_t),
    "n en train (tiempo)": cat_all.iloc[tr_t][y.iloc[tr_t] == 1].value_counts(),
    "n en test (tiempo)": cat_all.iloc[te_t][y.iloc[te_t] == 1].value_counts(),
}).fillna(0)
out["n en train (tiempo)"] = out["n en train (tiempo)"].astype(int)
out["n en test (tiempo)"] = out["n en test (tiempo)"].astype(int)
out.sort_values("n en test (tiempo)", ascending=False).round(3)

### 4.5 ¿Aprende atajos? Prueba con la IP
Tres modelos, en ambos repartos:
- **base**: solo comportamiento de la conexión (el modelo de arriba).
- **base + IP**: añade IP de origen y de destino.
- **solo IP**: no ve nada del tráfico, solo las IP.

Si "solo IP" detecta casi todo en el reparto al azar, hay un atajo disponible. Lo interesante es ver si ese atajo **sobrevive** al reparto por tiempo.

In [ ]:
IP_COLS = ["srcip", "dstip"]
variantes = {
    "base":        (NUM_COLS, CAT_COLS),
    "base + IP":   (NUM_COLS, CAT_COLS + IP_COLS),
    "solo IP":     ([], IP_COLS),
}
rows = []
for nombre, (nc, cc) in variantes.items():
    for split_name, (tr, te) in {"azar": (tr_r, te_r), "tiempo": (tr_t, te_t)}.items():
        p = fit_predict(nc, cc, tr, te)
        mm = metrics(y.iloc[te].values, p)
        rows.append(dict(modelo=nombre, reparto=split_name, precision=mm["precision"],
                         recall=mm["recall"], f1=mm["f1"], accuracy=mm["accuracy"]))
pd.DataFrame(rows).round(4).pivot(index="modelo", columns="reparto", values=["precision", "recall", "f1"]).round(4)

In [ ]:
# ¿Cuántas IP distintas hay y cuánta parte de los ataques cubren las pocas IP más habituales?
atk = df[df.Label == 1]
print("IP de origen distintas en ataques :", atk.srcip.nunique(), "| en tráfico normal:", df[df.Label == 0].srcip.nunique())
print("Top IP de origen en ataques (% de ataques):")
print((atk.srcip.value_counts(normalize=True).head(5) * 100).round(1))
print("\n¿Las IP de origen de ataque aparecen también en tráfico normal? ",
      len(set(atk.srcip) & set(df[df.Label == 0].srcip)), "IP compartidas")

## 5. Conclusiones (rellena tú)

Usa los números de las tablas de arriba. Guía:

1. **Hipótesis:** ¿ambas (recall y precisión) bajan con reparto por tiempo? ¿Más que `BIG_DIFF`? ¿En cuántos pliegues?
2. **Cuánto:** diferencia media en recall y en precisión, comparada con la variabilidad del reparto al azar.
3. **Por qué (fuga de datos):** % de gemelas exactas en azar vs tiempo (4.1), distancias (4.2), y qué pasa con la nota al quitar las gemelas (4.3).
4. **Por qué (cambio real):** tipos de ataque que se pierden (4.4) y diferencia de proporción de ataques entre bloques (sección 1.1). La precisión depende de la proporción de ataques: si cambia, la precisión cambia aunque el modelo sea igual de bueno.
5. **Atajos:** ¿qué pasa con "solo IP" al pasar de azar a tiempo? (4.5)
6. **Accuracy:** ¿cuánto supera al modelo tonto «todo normal»? (3.2)

**Limitaciones a nombrar en el post:** un solo dataset, un solo modelo, un solo umbral (0,5), submuestreo si `SAMPLE_FRAC < 1`, dos días de captura con un hueco entre ellos.